In [16]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Transforms
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # ResNet input size
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Dataset
dataset = datasets.ImageFolder(
    root="./data",
    transform=transform
)

print("Classes:", dataset.classes)
print("Number of images:", len(dataset))



Classes: ['cat', 'dog']
Number of images: 1000


In [17]:
# DataLoader
from torch.utils.data import random_split

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(
    dataset,
    [train_size, val_size]
)

In [18]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

In [19]:
print(len(train_loader.dataset))
print(len(val_loader.dataset))

800
200


In [20]:
from torchvision import models

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

print(model)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_sta

In [ ]:
import torch.nn as nn
import torch
for param in model.parameters():
    param.requires_grad = False

# Replace the final fully connected layer to match the number of classes in the dataset
model.fc = nn.Linear(model.fc.in_features, 2)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.fc.parameters(), lr=0.001)

for name, param in model.named_parameters():
    if param.requires_grad:
        print(name)

fc.weight
fc.bias


In [22]:
epochs = 5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("Training on device:", device)

Training on device: cpu


In [23]:
for epoch in range(epochs):

    model.train()
    epochloss = 0.0
    for images, labels in train_loader:  
        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epochloss += loss.item()
       

    print(f"Epoch [{epoch+1}/{epochs}], Loss: {epochloss/len(train_loader):.4f}")
    
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    print(f"Epoch [{epoch+1}/{epochs}], Validation Accuracy: {100 * correct / total:.2f}%")



Epoch [1/5], Loss: 0.2724
Epoch [1/5], Validation Accuracy: 99.00%
Epoch [2/5], Loss: 0.0475
Epoch [2/5], Validation Accuracy: 100.00%
Epoch [3/5], Loss: 0.0254
Epoch [3/5], Validation Accuracy: 100.00%
Epoch [4/5], Loss: 0.0254
Epoch [4/5], Validation Accuracy: 100.00%
Epoch [5/5], Loss: 0.0157
Epoch [5/5], Validation Accuracy: 100.00%


In [24]:
print(predicted[:10])
print(labels[:10])

tensor([1, 0, 1, 1, 0, 0, 0, 1])
tensor([1, 0, 1, 1, 0, 0, 0, 1])
